# Build 2 — Validation Harness and Logistic Regression Baseline

Kaggle Playground Series S6E8 — Predicting Smartphone Addiction

**Purpose:** run experiment **E001**, the first real modeling experiment,
using the `StratifiedKFold` validation harness and preprocessing pipeline
built in `src/validation.py` and `src/preprocessing.py`. Both modules
implement the frozen Build 1 decisions (see `docs/DECISIONS.md`):
`id` excluded, median imputation + standardization for numeric predictors,
explicit "Missing" category + one-hot encoding for categoricals,
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.

This notebook trains a real model and computes real cross-validated
ROC AUC — unlike Build 1, which was audit-only. It does not generate a
Kaggle submission (out of scope for this build per the linked issue).

## 1. Setup and data loading

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

from src.config import (
    CATEGORICAL_COLS,
    EXPERIMENTS_DIR,
    NUMERIC_COLS,
    RANDOM_SEED,
    TARGET_COLUMN,
    TRAIN_PATH,
)
from src.preprocessing import build_preprocessor
from src.validation import N_SPLITS, get_cv_splitter

pd.set_option("display.max_columns", 50)

train = pd.read_csv(TRAIN_PATH)
print("train shape:", train.shape)

train shape: (691369, 14)


In [2]:
FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS
X = train[FEATURE_COLS]
y = train[TARGET_COLUMN]

print("feature columns:", FEATURE_COLS)
print("target balance:", y.value_counts(normalize=True).round(4).to_dict())

feature columns: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact']
target balance: {1: 0.7094, 0: 0.2906}


## 2. E001 — Logistic Regression, raw/imputed features

**Hypothesis:** a linear model on the imputed, standardized, one-hot
encoded predictor set establishes a reasonable baseline, given that Build
1 found the target's strongest associations concentrated in a handful of
continuous "usage" features (Section 8 of `notebooks/01_eda.ipynb`).

**Method:** `Pipeline(preprocessor, LogisticRegression)`, evaluated with
the standard 5-fold `StratifiedKFold` harness. Each fold fits the
preprocessor only on that fold's training rows (via the pipeline), so
there is no leakage of validation-fold statistics into imputation or
scaling.

In [3]:
def run_cv_experiment(model, X: pd.DataFrame, y: pd.Series) -> dict:
    """Runs the standard CV harness for one sklearn-compatible model.

    Returns a dict with per-fold ROC AUC scores, their mean, and std.
    """
    pipeline = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("model", model),
        ]
    )
    splitter = get_cv_splitter()
    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(splitter.split(X, y), start=1):
        pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
        val_preds = pipeline.predict_proba(X.iloc[val_idx])[:, 1]
        fold_auc = roc_auc_score(y.iloc[val_idx], val_preds)
        fold_scores.append(fold_auc)
        print(f"fold {fold_idx}: ROC AUC = {fold_auc:.5f}")
    return {
        "fold_scores": fold_scores,
        "cv_mean": float(np.mean(fold_scores)),
        "cv_std": float(np.std(fold_scores)),
    }

In [4]:
t0 = time.time()
e001_model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
e001_results = run_cv_experiment(e001_model, X, y)
elapsed = time.time() - t0

print()
print(f"CV mean ROC AUC: {e001_results['cv_mean']:.5f}")
print(f"CV std:          {e001_results['cv_std']:.5f}")
print(f"elapsed:         {elapsed:.1f}s")

fold 1: ROC AUC = 0.91040


fold 2: ROC AUC = 0.91082


fold 3: ROC AUC = 0.91193


fold 4: ROC AUC = 0.91267


fold 5: ROC AUC = 0.91161

CV mean ROC AUC: 0.91149
CV std:          0.00081
elapsed:         25.3s


**Observation:** E001 (Logistic Regression, raw/imputed features)
scores a mean 5-fold ROC AUC of **0.9115** with a very small std
(**0.0008**) across folds — the CV estimate is stable, consistent with
Build 1's finding of no meaningful train/test shift or duplicate-leakage
risk (Sections 5 and 9 of `notebooks/01_eda.ipynb`). This is well above a
trivial baseline (a model with no signal would score ~0.50), and is
consistent with the strong univariate associations Build 1 found for the
screen-time feature family.

**Implication:** this is a credible, non-leaking baseline to build on. The
low fold-to-fold variance means small preprocessing or feature changes in
later experiments (E002-E005, proposed in Build 1) should be evaluated
against this same 5-fold harness for a fair comparison.

## 3. Recording the experiment

Per project convention (`docs/DECISIONS.md`), experiment IDs are assigned
only when an experiment actually runs. E001 has now run, so its result is
appended to `experiments/experiments.csv` as a real row — not a
placeholder.

In [5]:
import csv
from datetime import date

experiments_path = EXPERIMENTS_DIR / "experiments.csv"

row = {
    "experiment_id": "E001",
    "date": date.today().isoformat(),
    "model": "LogisticRegression",
    "feature_set": "raw_predictors",
    "hypothesis": (
        "A linear baseline on imputed/standardized/one-hot predictors "
        "should score well given Build 1's finding that target signal is "
        "concentrated in a few continuous screen-time features."
    ),
    "preprocessing": (
        "median imputation + StandardScaler (numeric); "
        "explicit Missing category + OneHotEncoder (categorical); id excluded"
    ),
    "cv_method": f"StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_SEED})",
    "seed": RANDOM_SEED,
    "parameters": "LogisticRegression(max_iter=1000, random_state=42)",
    "fold_1_auc": round(e001_results["fold_scores"][0], 5),
    "fold_2_auc": round(e001_results["fold_scores"][1], 5),
    "fold_3_auc": round(e001_results["fold_scores"][2], 5),
    "fold_4_auc": round(e001_results["fold_scores"][3], 5),
    "fold_5_auc": round(e001_results["fold_scores"][4], 5),
    "cv_mean": round(e001_results["cv_mean"], 5),
    "cv_std": round(e001_results["cv_std"], 5),
    "public_lb": "",
    "submission_file": "",
    "conclusion": (
        "Stable, credible linear baseline (mean 0.9115, std 0.0008). "
        "Confirms Build 1's leakage/shift findings translate into a "
        "well-behaved CV estimate. Candidate for comparison target for "
        "E002-E005 and Build 3 strong-model benchmarks."
    ),
    "next_action": (
        "Run E002 (missing indicators), E003 (screen-time component "
        "features), E004 (ordinal stress_level), E005 (non-linear sanity "
        "check) against the same harness before moving to Build 3."
    ),
}

existing = pd.read_csv(experiments_path)
if "E001" in existing["experiment_id"].astype(str).values:
    print("E001 already recorded in experiments.csv — skipping append (idempotent re-run).")
else:
    with open(experiments_path, "r", newline="", encoding="utf-8") as f:
        fieldnames = next(csv.reader(f))
    with open(experiments_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)
    print("Row appended to", experiments_path)

E001 already recorded in experiments.csv — skipping append (idempotent re-run).


In [6]:
pd.read_csv(experiments_path)

,experiment_id,date,model,feature_set,hypothesis,preprocessing,cv_method,seed,parameters,fold_1_auc,fold_2_auc,fold_3_auc,fold_4_auc,fold_5_auc,cv_mean,cv_std,public_lb,submission_file,conclusion,next_action
0,E001,2026-08-17,LogisticRegression,raw_predictors,A linear baseline on imputed/standardized/one-...,median imputation + StandardScaler (numeric); ...,"StratifiedKFold(n_splits=5, shuffle=True, rand...",42,"LogisticRegression(max_iter=1000, random_state...",0.9104,0.91082,0.91193,0.91267,0.91161,0.91149,0.00081,NaN,NaN,"Stable, credible linear baseline (mean 0.9115,...","Run E002 (missing indicators), E003 (screen-ti..."


## 4. Build 2 findings summary

| Finding | Evidence | Implication |
|---|---|---|
| E001 Logistic Regression baseline scores mean ROC AUC 0.9115 (std 0.0008) | Section 2 above | Credible, stable baseline; strong signal from the screen-time feature family carries through into an actual model |
| CV harness and preprocessing implemented in `src/` are reusable | `src/validation.py`, `src/preprocessing.py`, `tests/test_validation.py`, `tests/test_preprocessing.py` | Build 3+ can reuse `get_cv_splitter()` and `build_preprocessor()` directly, or extend them, rather than re-deriving the pipeline per notebook |
| No leakage surfaced during actual model fitting (each fold fits its own preprocessor) | `run_cv_experiment` fits `Pipeline` per fold | Validates the Build 1 assumption that plain `StratifiedKFold` is trustworthy for this dataset |

Per the linked issue, this build stops at a validated baseline — no
hyperparameter tuning, no strong-model benchmarking, and no Kaggle
submission. Those are Build 3+.